# RoadSense: Robust Traffic Sign Recognition
## Phase 1: Exploratory Data Analysis (EDA) & Leakage-Free Validation Split
**Group 4 | Atria University**  
- **Team Leader:** Aseem  
- **Members:** Raju, Sohail, Manoj  

---
### Objectives of Phase 1:
1. **Inspect Class Distribution:** Quantify frequency of all 43 classes and determine class imbalance.
2. **Visual Inspection:** Display a sample grid of all 43 traffic sign categories with readable names.
3. **Resolution & Brightness Analysis:** Evaluate the distribution of image widths, heights, aspect ratios, luminance, and contrast.
4. **Disjoint Track-Based Splitting:** Implement a leak-free train/validation split by partitioning continuous video sequences (tracks).
5. **Label-Safe Augmentation Discussion:** Categorize augmentations that preserve vs alter traffic sign semantic labels.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Add project root to path
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
from src.utils import CLASS_NAMES, CLASS_CATEGORIES, set_seed

set_seed(42)
sns.set_theme(style="whitegrid")
DATA_DIR = ROOT_DIR / "data"
FIG_DIR = ROOT_DIR / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Environment initialized. Root directory:", ROOT_DIR)

### 1. Load Metadata and Verify Dataset
We load `Train.csv` and `Test.csv`. In GTSRB, the filename contains three underscore-separated numbers:  
`[ClassId]_[TrackId]_[FrameId].png`  
The `TrackId` is critical: all 30 frames within a track belong to the same physical sign recorded continuously as the car approaches.

In [ ]:
df_train = pd.read_csv(DATA_DIR / "Train.csv")
df_test = pd.read_csv(DATA_DIR / "Test.csv")

# Parse track_id and frame_id
df_train["filename"] = df_train["Path"].apply(lambda p: Path(p).name)
df_train["track_id"] = df_train["filename"].apply(lambda fn: fn.split("_")[1] if len(fn.split("_")) > 1 else "0")
df_train["frame_id"] = df_train["filename"].apply(lambda fn: fn.split("_")[2].split(".")[0] if len(fn.split("_")) > 2 else "0")
df_train["class_name"] = df_train["ClassId"].map(CLASS_NAMES)

print(f"Total Training Images: {len(df_train):,}")
print(f"Total Test Images:     {len(df_test):,}")
print(f"Total Unique Classes:  {df_train['ClassId'].nunique()}")
df_train.head(3)

### 2. Class Distribution & Imbalance Analysis
GTSRB exhibits substantial real-world class imbalance because certain speed limit and priority signs are far more frequent than rare hazard signs.

In [ ]:
train_counts = df_train["ClassId"].value_counts().sort_index()
test_counts = df_test["ClassId"].value_counts().sort_index()

max_cls = train_counts.idxmax()
min_cls = train_counts.idxmin()
imbalance_ratio = train_counts.max() / train_counts.min()

print(f"Most frequent sign:  Class {max_cls} ({CLASS_NAMES[max_cls]}) with {train_counts.max()} images")
print(f"Least frequent sign: Class {min_cls} ({CLASS_NAMES[min_cls]}) with {train_counts.min()} images")
print(f"Class Imbalance Ratio: {imbalance_ratio:.2f}x")

plt.figure(figsize=(15, 6))
x = np.arange(43)
w = 0.4
plt.bar(x - w/2, train_counts.values, w, label=f"Train ({len(df_train):,})", color="#2b5c8f")
plt.bar(x + w/2, test_counts.values, w, label=f"Test ({len(df_test):,})", color="#e07a5f")
plt.xlabel("Class ID (0 to 42)")
plt.ylabel("Number of Images")
plt.title("Class Distribution in GTSRB: Showing Severe Class Imbalance (10.71x)")
plt.xticks(x, [str(i) for i in range(43)], rotation=0)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=300)
plt.show()

### 3. Visual Grid of All 43 Traffic Sign Categories

In [ ]:
fig, axes = plt.subplots(7, 7, figsize=(16, 16))
axes = axes.flatten()

for cid in range(43):
    ax = axes[cid]
    class_rows = df_train[df_train["ClassId"] == cid]
    sample_path = DATA_DIR / class_rows.iloc[len(class_rows)//2]["Path"]
    img = Image.open(sample_path)
    ax.imshow(img)
    title = f"[{cid}] {CLASS_NAMES[cid]}"
    if len(title) > 22:
        title = title[:20] + "..."
    ax.set_title(title, fontsize=8)
    ax.axis("off")

for idx in range(43, len(axes)):
    axes[idx].axis("off")

plt.suptitle("Sample Images from All 43 GTSRB Classes", fontsize=14, y=0.99)
plt.tight_layout()
plt.savefig(FIG_DIR / "sample_grid_43_classes.png", dpi=300)
plt.show()

### 4. Image Dimensions & Brightness Distributions
Traffic signs vary considerably in distance from the camera (from 25x25 up to 243x225 pixels), but maintain an almost exact 1:1 aspect ratio. Resizing to a uniform square (e.g. 32x32 or 48x48) is thus strongly justified.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_train["Width"], bins=35, kde=True, ax=axs[0], color="#2b5c8f", label="Width")
sns.histplot(df_train["Height"], bins=35, kde=True, ax=axs[0], color="#81b29a", label="Height", alpha=0.6)
axs[0].set_title("Image Width & Height Distribution")
axs[0].set_xlabel("Pixels")
axs[0].legend()

axs[1].scatter(df_train["Width"], df_train["Height"], alpha=0.15, color="#3d405b", s=10)
axs[1].plot([15, 250], [15, 250], "--r", label="1:1 Aspect Ratio")
axs[1].set_title("Width vs Height Correlation")
axs[1].set_xlabel("Width (px)")
axs[1].set_ylabel("Height (px)")
axs[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "image_size_distribution.png", dpi=300)
plt.show()

### 5. Leakage-Free Validation Split by Track
> **CRITICAL ARCHITECTURAL REQUIREMENT:**  
> GTSRB images come in continuous 30-frame track sequences of the same sign. Randomly splitting individual images causes the model to memorize physical instances, causing severe **data leakage** and invalid validation results.  
> We split **atomic tracks** (80% train tracks, 20% validation tracks per class).

In [ ]:
train_splits = []
val_splits = []

for class_id, class_group in df_train.groupby("ClassId"):
    unique_tracks = class_group["track_id"].unique()
    shuffled_tracks = np.random.permutation(unique_tracks)
    n_val = max(1, int(np.round(len(unique_tracks) * 0.20)))
    val_tracks = set(shuffled_tracks[:n_val])
    train_tracks = set(shuffled_tracks[n_val:])
    
    train_splits.append(class_group[class_group["track_id"].isin(train_tracks)])
    val_splits.append(class_group[class_group["track_id"].isin(val_tracks)])

train_df = pd.concat(train_splits).reset_index(drop=True)
val_df = pd.concat(val_splits).reset_index(drop=True)

# Verify zero leakage
for cid in range(43):
    t_tracks = set(train_df[train_df["ClassId"] == cid]["track_id"].unique())
    v_tracks = set(val_df[val_df["ClassId"] == cid]["track_id"].unique())
    assert len(t_tracks.intersection(v_tracks)) == 0

print(f"Train Split: {len(train_df):,} images ({len(train_df)/len(df_train)*100:.1f}%)")
print(f"Val Split:   {len(val_df):,} images ({len(val_df)/len(df_train)*100:.1f}%)")
print("Leakage check: 0 overlapping tracks between Train and Val sets!")

train_df.to_csv(DATA_DIR / "train_split.csv", index=False)
val_df.to_csv(DATA_DIR / "val_split.csv", index=False)
print("Split CSVs saved successfully.")

### 6. Discussion: Label-Safe vs Dangerous Data Augmentations
The project brief highlights: *"Careful with augmentation: horizontally flipping a 'keep left' sign turns it into a 'keep right' sign. Discuss which augmentations are label-safe."*

| Augmentation | Status | Reason & Safety Constraint |
| :--- | :--- | :--- |
| **Horizontal Flip** | ❌ **DANGEROUS** | Turns *Keep Left* (Class 39) into *Keep Right* (Class 38); turns *Curve Left* (Class 19) into *Curve Right* (Class 20). Must **not** be used blindly. |
| **Vertical Flip** | ❌ **DANGEROUS** | Inverts directional arrows and inverted triangles (e.g. *Yield*). Must **never** be used. |
| **Small Rotation (±10°)** |  **LABEL-SAFE** | Simulates vehicle camera pitch/roll variations without altering sign semantics. |
| **Color Jitter (Brightness/Contrast)** |  **LABEL-SAFE** | Simulates morning glare, overcast skies, sunset, and night driving conditions. |
| **Gaussian Noise** |  **LABEL-SAFE** | Simulates sensor grain in low-light camera settings. |
| **Motion Blur / Gaussian Blur** |  **LABEL-SAFE** | Simulates vehicle movement and dirty lenses. |
| **Random Cutout / Occlusion** |  **LABEL-SAFE** | Simulates signs partially blocked by branches, lampposts, or stickers. |